# Runbook 1.3.c: Jamming / denial-of-service attack  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Analyze Risk · **Maps to:** 1.3.B · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** A wireless intrusion detection system samples the noise floor of a Wi-Fi channel once per
second. Each sample is an integer noise measurement (higher = more noise). Your job is to
detect whether the sample series contains a sustained jamming event.

**Detection rule:** a series indicates jamming if it contains a run of at least **N = 5**
consecutive samples that are ALL strictly greater than **T = 80**.

These constants are fixed for this problem (see the Reference section below). A single spike, or a short
burst of noise below the window, is not considered jamming.

**Lesson objective.** Explain how radio-frequency jamming causes a wireless denial-of-service, describe what a sustained jamming event looks like in signal-strength monitoring data, and implement a detector that flags sustained interference in a sample series.

### What comes in
A list of integers (noise-floor samples). May be empty. No length limit. Values >= 0.

### What must come out
`True` (jamming detected) or `False` (no sustained jamming).

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```
[20, 25, 90, 22, 30, 18, 24]            -> False  (only one spike; run length 1 < N=5)
[15, 82, 85, 88, 91, 83, 87, 20]        -> True   (positions 1-6 all > 80; run = 6 >= N=5)
[70, 72, 75, 71, 73, 74, 76]            -> False   (none exceed T=80)
[50, 81, 82, 83, 84, 85, 86, 87, 88, 30]-> True   (positions 1-8 all > 80; run = 8 >= N=5)
[]                                       -> False  (empty input)
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] T = 80 (strictly greater than), N = 5 (minimum consecutive samples above T).
- [ ] Deterministic, offline, standard library only.
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(samples: list[int]) -> bool
```

#### Input (`samples`)
A list of integers, each representing one noise-floor measurement from a wireless monitor.
Values are >= 0. The list may be empty.

#### Output (`result`)
`True` if the series contains a sustained jamming event; `False` otherwise.

#### Detection rule
**Threshold T = 80 (strictly greater than). Minimum consecutive window N = 5.**

A series is flagged as jamming if it contains a run of at least N = 5 consecutive samples
where every sample in the run is strictly greater than T = 80.

A run is interrupted as soon as any sample is <= T. The run counter resets to zero.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- T and N are fixed constants (T=80, N=5); do not parameterize them.
- An empty list returns False.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
from typing import List

THRESHOLD = 80
WINDOW = 5


def solve(samples: List[int]) -> bool:
    run = 0
    for s in samples:
        if s > THRESHOLD:
            run += 1
            if run >= WINDOW:
                return True
        else:
            run = 0
    return False


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_single_spike_not_jamming():
    assert impl.solve([20, 25, 90, 22, 30, 18, 24]) is False


def test_run_of_six_is_jamming():
    assert impl.solve([15, 82, 85, 88, 91, 83, 87, 20]) is True


def test_all_below_threshold():
    assert impl.solve([70, 72, 75, 71, 73, 74, 76]) is False


def test_run_of_nine_is_jamming():
    assert impl.solve([50, 81, 82, 83, 84, 85, 86, 87, 88, 30]) is True


def test_empty_input():
    assert impl.solve([]) is False


def test_run_of_exactly_five():
    assert impl.solve([10, 81, 82, 83, 84, 85, 10]) is True


def test_run_of_four_not_jamming():
    assert impl.solve([10, 81, 82, 83, 84, 10]) is False


def test_threshold_boundary_not_strictly_greater():
    assert impl.solve([80, 80, 80, 80, 80, 80]) is False


def test_threshold_boundary_81_counts():
    assert impl.solve([81, 81, 81, 81, 81]) is True


def test_interrupted_run_not_jamming():
    assert impl.solve([81, 82, 83, 84, 80, 81, 82, 83, 84]) is False


def test_long_clean_then_jamming():
    clean = [10, 20, 30] * 10
    jammed = [85, 86, 87, 88, 89]
    assert impl.solve(clean + jammed) is True


def test_jamming_at_start():
    assert impl.solve([90, 91, 92, 93, 94, 10, 10]) is True


def test_single_element_below():
    assert impl.solve([50]) is False


def test_single_element_above():
    assert impl.solve([99]) is False


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Call `solve` on an input you make up, and predict the answer first.

- [ ] Your prediction matched

## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it